# Day 6 — Python: API Response Handling

> **Real APIs used (free, no key required):**
> - **Open-Meteo** — weather forecast data
> - **JSONPlaceholder** — fake REST (posts, users, todos)
> - **REST Countries** — country info
>
> **Topics:** `requests` · Pagination · Throttling · Retry + Exponential Backoff · Flatten nested JSON  
> **Run order:** top to bottom — requires internet connection

In [27]:
import requests
import time
from pprint import pprint

# All retry logic + mock helpers live in api_helpers.py — open it once to read the internals
# Here we just import and use them
from api_helpers import get_with_retry, make_429, make_500, make_200, fake_sleep

print("Ready.")

Ready.


---
## 1. Basic GET Request — JSONPlaceholder

JSONPlaceholder is a free fake REST API perfect for learning.  
https://jsonplaceholder.typicode.com

In [28]:
# GET a single post
resp = requests.get(
    'https://jsonplaceholder.typicode.com/posts/1',
    timeout=10,
)

print('Status code:', resp.status_code)
print('Content-Type:', resp.headers.get('Content-Type'))
print('Response size:', len(resp.content), 'bytes')

post = resp.json()     # parse JSON → dict
pprint(post)

Status code: 200
Content-Type: application/json; charset=utf-8
Response size: 292 bytes
{'body': 'quia et suscipit\n'
         'suscipit recusandae consequuntur expedita et cum\n'
         'reprehenderit molestiae ut ut quas totam\n'
         'nostrum rerum est autem sunt rem eveniet architecto',
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio '
          'reprehenderit',
 'userId': 1}


In [29]:
# raise_for_status() — always use this before consuming the response
resp.raise_for_status()     # does nothing on 200; raises HTTPError on 4xx/5xx
print('Status is OK.')

# What does a 404 look like?
bad_resp = requests.get('https://jsonplaceholder.typicode.com/posts/9999', timeout=10)
print('\n404 status:', bad_resp.status_code)
try:
    bad_resp.raise_for_status()
except requests.exceptions.HTTPError as e:
    print('HTTPError caught:', e)

Status is OK.

404 status: 404
HTTPError caught: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/9999


In [30]:
# Use params= dict — NOT f-string URL (requests handles URL encoding for you)
resp = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params={'userId': 1},    # → /posts?userId=1
    timeout=10,
)
posts_user1 = resp.json()
print(f'User 1 has {len(posts_user1)} posts')
print('First post title:', posts_user1[0]['title'])

User 1 has 10 posts
First post title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit


---
## 2. Pagination — Fetch All Pages

JSONPlaceholder supports `_page` and `_limit` query params.  
There are 100 posts total — we fetch them in pages of 10.

In [31]:
# Manual page loop — stop when the page comes back empty
PAGE_SIZE = 10
DELAY_SEC = 0.3      # throttle: 0.3s between pages → max ~3 req/sec

all_posts = []
page = 1

while True:
    resp = requests.get(
        'https://jsonplaceholder.typicode.com/posts',
        params={'_page': page, '_limit': PAGE_SIZE},
        timeout=10,
    )
    resp.raise_for_status()
    batch = resp.json()

    if not batch:          # empty page → no more data
        break

    all_posts.extend(batch)
    print(f'Page {page:2d}: fetched {len(batch):3d} posts (total so far: {len(all_posts)})')
    page += 1
    time.sleep(DELAY_SEC)  # throttle

print(f'\nDone. Total posts: {len(all_posts)}')

Page  1: fetched  10 posts (total so far: 10)
Page  2: fetched  10 posts (total so far: 20)
Page  3: fetched  10 posts (total so far: 30)
Page  4: fetched  10 posts (total so far: 40)
Page  5: fetched  10 posts (total so far: 50)
Page  6: fetched  10 posts (total so far: 60)
Page  7: fetched  10 posts (total so far: 70)
Page  8: fetched  10 posts (total so far: 80)
Page  9: fetched  10 posts (total so far: 90)
Page 10: fetched  10 posts (total so far: 100)

Done. Total posts: 100


In [32]:
# Generator pattern — yields one page at a time (lazy, memory-efficient)
def paginate_api(base_url, page_size=10, delay=0.3):
    """Yields successive pages from a JSONPlaceholder-style endpoint."""
    page = 1
    while True:
        resp = requests.get(
            base_url,
            params={'_page': page, '_limit': page_size},
            timeout=10,
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            return
        yield batch
        page += 1
        time.sleep(delay)

# Caller: collect exactly the first 3 pages
records = []
for page_num, page_data in enumerate(paginate_api(
    'https://jsonplaceholder.typicode.com/todos', page_size=5
), start=1):
    records.extend(page_data)
    print(f'Page {page_num}: +{len(page_data)} todos')
    if page_num >= 3:
        break

print(f'Collected {len(records)} todos from 3 pages')

Page 1: +5 todos
Page 2: +5 todos
Page 3: +5 todos
Collected 15 todos from 3 pages


---
## 3. Retry Logic with Exponential Backoff

Retry on server errors (5xx) and rate limits (429).  
Raise immediately on client errors (4xx) — those are our bug.

**Backoff schedule (base_delay=1):** 1s → 2s → 4s → 8s

In [33]:
# get_with_retry is defined in api_helpers.py — open that file to read how it works.
# Here we just verify it works on a real endpoint:

data = get_with_retry('https://jsonplaceholder.typicode.com/users/1')
print(f"Fetched user: {data['name']} ({data['email']})")

Fetched user: Leanne Graham (Sincere@april.biz)


In [34]:
# Simulate what happens on a 404 (client error — should NOT retry)
try:
    get_with_retry('https://jsonplaceholder.typicode.com/posts/9999')
except requests.exceptions.HTTPError as e:
    print(f'Client error raised immediately (no retry): {e}')

Client error raised immediately (no retry): 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/9999


---
## 3b. Retry in Action — Watch It Recover

Three real scenarios simulated with `unittest.mock` so responses are **controlled and instant**  
(no real server needed, no actual waiting — `time.sleep` is patched to just print what it *would* wait).

| Scenario | Attempts | What happens |
|----------|----------|--------------|
| A | 429 → 429 → 200 | Rate limited twice, then succeeds |
| B | 500 → 500 → 200 | Server errors, exponential backoff, then succeeds |
| C | 429 × 4 | All retries exhausted → RuntimeError |

In [36]:
from unittest.mock import patch

# make_429 / make_500 / make_200 / fake_sleep are imported from api_helpers
# — open api_helpers.py to see how they build fake HTTP responses

# ── SCENARIO A: 429 → 429 → 200 ──────────────────────────────────────────────
# API rate-limits us twice, then clears the limit on the third attempt.

fake_responses_A = [
    make_429(retry_after=2),   # attempt 0 → 429, told to wait 2s
    make_429(retry_after=2),   # attempt 1 → still rate limited
    make_200({'id': 42, 'title': 'Data engineering is fun', 'userId': 1}),  # attempt 2 → 200 ✓
]
sleep_log_A = []

print('━' * 55)
print('SCENARIO A  —  429 → 429 → 200 (rate limit, then OK)')
print('━' * 55)

with patch('requests.get', side_effect=fake_responses_A), \
     patch('time.sleep',   side_effect=lambda s: sleep_log_A.append(round(s, 1)) or
                                                  fake_sleep(s)):
    result = get_with_retry('https://api.example.com/posts/42', max_retries=4, base_delay=1.0)

print()
print('✓ SUCCESS after 2 retries')
print('  Response :', result)
print('  Slept (s):', sleep_log_A)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCENARIO A  —  429 → 429 → 200 (rate limit, then OK)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [429] Rate limited — waiting 2.0s (attempt 1/4)
        ↳ [would sleep 2.00s in production — skipped in demo]
  [429] Rate limited — waiting 2.0s (attempt 2/4)
        ↳ [would sleep 2.00s in production — skipped in demo]

✓ SUCCESS after 2 retries
  Response : {'id': 42, 'title': 'Data engineering is fun', 'userId': 1}
  Slept (s): [2.0, 2.0]


In [37]:
# ── SCENARIO B: 500 → 500 → 200 (exponential backoff + jitter) ───────────────
# Server is overloaded on the first two calls, then recovers.
# Backoff grows: ~1s → ~2s → success. Jitter prevents thundering herd.

fake_responses_B = [
    make_500(),   # attempt 0 → 500, backoff ≈ 1s + jitter
    make_500(),   # attempt 1 → 500, backoff ≈ 2s + jitter
    make_200({'id': 7, 'status': 'ok', 'records': 150}),   # attempt 2 → 200 ✓
]
sleep_log_B = []

print('━' * 55)
print('SCENARIO B  —  500 → 500 → 200 (server error → backoff → OK)')
print('━' * 55)

with patch('requests.get', side_effect=fake_responses_B), \
     patch('time.sleep',   side_effect=lambda s: sleep_log_B.append(round(s, 2)) or
                                                  fake_sleep(s)):
    result = get_with_retry('https://api.example.com/data', max_retries=4, base_delay=1.0)

print()
print('✓ SUCCESS after 2 retries')
print('  Response  :', result)
print('  Slept (s) :', sleep_log_B)
print('  Note: each wait > previous (exponential), tiny random jitter added')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCENARIO B  —  500 → 500 → 200 (server error → backoff → OK)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [500] Server error — retrying in 1.24s (attempt 1/4)
        ↳ [would sleep 1.24s in production — skipped in demo]
  [500] Server error — retrying in 2.44s (attempt 2/4)
        ↳ [would sleep 2.44s in production — skipped in demo]

✓ SUCCESS after 2 retries
  Response  : {'id': 7, 'status': 'ok', 'records': 150}
  Slept (s) : [1.24, 2.44]
  Note: each wait > previous (exponential), tiny random jitter added


In [38]:
# ── SCENARIO C: all 4 attempts return 429 → RuntimeError raised ──────────────
# API never clears the rate limit within our retry budget → give up.
# In production: write to dead-letter queue / alert on-call.

all_429_C = [make_429(retry_after=1)] * 4
sleep_log_C = []

print('━' * 55)
print('SCENARIO C  —  429 × 4  (all retries exhausted → RuntimeError)')
print('━' * 55)

with patch('requests.get', side_effect=all_429_C), \
     patch('time.sleep',   side_effect=lambda s: sleep_log_C.append(round(s, 1)) or
                                                  fake_sleep(s)):
    try:
        get_with_retry('https://api.example.com/data', max_retries=4, base_delay=1.0)
    except RuntimeError as e:
        print()
        print(f'✗ RuntimeError: {e}')
        print(f'  Total sleeps logged: {sleep_log_C}')
        print()
        print('  Production response:')
        print('    • Write record to dead-letter Kafka topic / S3 path')
        print('    • Emit metric to Datadog / CloudWatch')
        print('    • PagerDuty alert if failure rate > threshold')

print()
print('━' * 55)
print('SUMMARY — what get_with_retry does per status code:')
print('━' * 55)
rows = [
    ('429',              'Retry-After header',        'retry after header wait'),
    ('500 / 502 / 503',  'base × 2^attempt + jitter', 'retry with backoff'),
    ('4xx (not 429)',    'none',                      'raise immediately — our bug'),
    ('Timeout',          'base × 2^attempt',          'retry with backoff'),
    ('ConnectionError',  'base × 2^attempt',          'retry with backoff'),
    ('All retries done', 'n/a',                       'raise RuntimeError'),
]
print(f'  {"Status":<22} {"Wait":<28} {"Action"}')
print(f'  {"-"*22} {"-"*28} {"-"*25}')
for status, wait, action in rows:
    print(f'  {status:<22} {wait:<28} {action}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SCENARIO C  —  429 × 4  (all retries exhausted → RuntimeError)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [429] Rate limited — waiting 1.0s (attempt 1/4)
        ↳ [would sleep 1.00s in production — skipped in demo]
  [429] Rate limited — waiting 1.0s (attempt 2/4)
        ↳ [would sleep 1.00s in production — skipped in demo]
  [429] Rate limited — waiting 1.0s (attempt 3/4)
        ↳ [would sleep 1.00s in production — skipped in demo]
  [429] Rate limited — waiting 1.0s (attempt 4/4)
        ↳ [would sleep 1.00s in production — skipped in demo]

✗ RuntimeError: All 4 retries exhausted for https://api.example.com/data
  Total sleeps logged: [1.0, 1.0, 1.0, 1.0]

  Production response:
    • Write record to dead-letter Kafka topic / S3 path
    • Emit metric to Datadog / CloudWatch
    • PagerDuty alert if failure rate > threshold

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SUMMARY — what get_with_retry 

---
## 4. Throttling — Rate Limit Compliance

Fetch weather for 5 cities from Open-Meteo API with a fixed delay between requests.

In [39]:
# Open-Meteo: free weather API, no key required
# https://open-meteo.com/en/docs

CITIES = [
    {'name': 'Delhi',    'lat': 28.6139,  'lon': 77.2090},
    {'name': 'Mumbai',   'lat': 19.0760,  'lon': 72.8777},
    {'name': 'London',   'lat': 51.5074,  'lon': -0.1278},
    {'name': 'New York', 'lat': 40.7128,  'lon': -74.0060},
    {'name': 'Tokyo',    'lat': 35.6762,  'lon': 139.6503},
]

WEATHER_URL  = 'https://api.open-meteo.com/v1/forecast'
RATE_DELAY   = 0.5    # 0.5s between calls → max 2 req/sec

weather_records = []

for city in CITIES:
    data = get_with_retry(
        WEATHER_URL,
        params={
            'latitude':      city['lat'],
            'longitude':     city['lon'],
            'current':       ['temperature_2m', 'wind_speed_10m', 'relative_humidity_2m'],
            'timezone':      'auto',
            'forecast_days': 1,
        },
    )
    current = data['current']
    record = {
        'city':        city['name'],
        'temp_c':      current.get('temperature_2m'),
        'wind_kmh':    current.get('wind_speed_10m'),
        'humidity_pct': current.get('relative_humidity_2m'),
        'fetched_at':  current.get('time'),
    }
    weather_records.append(record)
    print(f"{city['name']:10s}: {record['temp_c']}°C  wind {record['wind_kmh']} km/h  humidity {record['humidity_pct']}%")
    time.sleep(RATE_DELAY)   # throttle

print(f'\nFetched weather for {len(weather_records)} cities.')

Delhi     : 38.3°C  wind 11.9 km/h  humidity 35%
Mumbai    : 26.6°C  wind 9.7 km/h  humidity 88%
London    : 24.4°C  wind 17.6 km/h  humidity 45%
New York  : 23.1°C  wind 1.8 km/h  humidity 77%
Tokyo     : 20.5°C  wind 2.6 km/h  humidity 97%

Fetched weather for 5 cities.


---
## 5. Flatten Nested JSON — Posts + User Enrichment

Fetch all posts, then enrich each with the author's name by fetching the user.  
Classic DE pattern: join an API stream with a lookup.

In [40]:
# Step 1: fetch posts (first page only for demo)
posts = get_with_retry(
    'https://jsonplaceholder.typicode.com/posts',
    params={'_page': 1, '_limit': 5},
)
print(f'Fetched {len(posts)} posts')

# Step 2: build a user cache to avoid re-fetching the same user repeatedly
user_cache = {}

def get_user(user_id):
    if user_id not in user_cache:
        user_cache[user_id] = get_with_retry(
            f'https://jsonplaceholder.typicode.com/users/{user_id}'
        )
        time.sleep(0.2)   # throttle
    return user_cache[user_id]

# Step 3: flatten posts + enrich with user info
flat_records = []
for post in posts:
    user = get_user(post['userId'])
    flat_records.append({
        'post_id':    post['id'],
        'title':      post['title'][:40],
        'user_id':    post['userId'],
        'user_name':  user['name'],
        'user_email': user['email'],
        'user_city':  user['address']['city'],
    })

print(f'\nEnriched {len(flat_records)} posts with user info:')
for r in flat_records:
    print(f"  post {r['post_id']:3d} | {r['user_name']:20s} | {r['user_city']}")

Fetched 5 posts

Enriched 5 posts with user info:
  post   1 | Leanne Graham        | Gwenborough
  post   2 | Leanne Graham        | Gwenborough
  post   3 | Leanne Graham        | Gwenborough
  post   4 | Leanne Graham        | Gwenborough
  post   5 | Leanne Graham        | Gwenborough


---
## 6. REST Countries — Flatten Deep Nesting

REST Countries returns very deeply nested JSON — great practice for real-world flattening.

In [41]:
# Fetch all countries — one big response (~600KB)
# https://restcountries.com
resp = requests.get('https://restcountries.com/v3.1/region/asia', timeout=15)
resp.raise_for_status()
countries = resp.json()
print(f'Fetched {len(countries)} Asian countries')

# Example raw structure for one country:
sample = countries[0]
print('\nSample keys:', list(sample.keys())[:10])
print('name structure:', sample.get('name'))
print('capital:', sample.get('capital'))
print('currencies:', sample.get('currencies'))

Fetched 3 Asian countries


KeyError: 0

In [ ]:
# Flatten with safe .get() for missing fields — many countries have no capital, currency, etc.
def flatten_country(c):
    currencies = c.get('currencies', {})
    currency_codes = list(currencies.keys())           # e.g. ['INR']
    currency_name  = list(currencies.values())[0].get('name', 'N/A') if currencies else 'N/A'

    languages = c.get('languages', {})
    lang_list = ', '.join(languages.values()) if languages else 'N/A'

    return {
        'name':        c['name']['common'],
        'official':    c['name']['official'],
        'region':      c.get('region', 'N/A'),
        'subregion':   c.get('subregion', 'N/A'),
        'capital':     c.get('capital', ['N/A'])[0] if c.get('capital') else 'N/A',
        'population':  c.get('population', 0),
        'area_km2':    c.get('area', 0),
        'currency':    currency_codes[0] if currency_codes else 'N/A',
        'currency_name': currency_name,
        'languages':   lang_list,
        'timezones':   c.get('timezones', []),
    }

flat_countries = [flatten_country(c) for c in countries]

# Sort by population descending
flat_countries.sort(key=lambda x: x['population'], reverse=True)

print(f'Flattened {len(flat_countries)} countries. Top 10 by population:')
for c in flat_countries[:10]:
    print(f"  {c['name']:20s}  pop: {c['population']:>12,}  capital: {c['capital']}")

---
## 7. Error Handling — Mixed API Responses

Simulate a batch of API calls where some fail. Never drop failed records silently.

In [42]:
# Fetch 10 specific post IDs — some exist (1-100), some don't (9998, 9999)
POST_IDS = [1, 2, 3, 9998, 5, 9999, 7, 8]

valid_posts   = []
failed_fetches = []

for post_id in POST_IDS:
    try:
        resp = requests.get(
            f'https://jsonplaceholder.typicode.com/posts/{post_id}',
            timeout=10,
        )
        resp.raise_for_status()   # raises HTTPError on 404
        valid_posts.append(resp.json())

    except requests.exceptions.HTTPError as e:
        failed_fetches.append({'id': post_id, 'error': str(e), 'type': 'HTTPError'})

    except requests.exceptions.Timeout:
        failed_fetches.append({'id': post_id, 'error': 'Timeout', 'type': 'Timeout'})

    except requests.exceptions.ConnectionError as e:
        failed_fetches.append({'id': post_id, 'error': str(e), 'type': 'ConnectionError'})

    time.sleep(0.2)

print(f'Fetched:  {len(valid_posts)} posts')
print(f'Failed:   {len(failed_fetches)} records → would go to dead-letter table')
print('\nFailed records:')
for r in failed_fetches:
    print(f"  post_id={r['id']:5d} | {r['type']} | {r['error'][:60]}")

Fetched:  6 posts
Failed:   2 records → would go to dead-letter table

Failed records:
  post_id= 9998 | HTTPError | 404 Client Error: Not Found for url: https://jsonplaceholder
  post_id= 9999 | HTTPError | 404 Client Error: Not Found for url: https://jsonplaceholder


---
## 8. Production Pattern — Paginated + Retry + Throttle Combined

In [43]:
def fetch_all_paginated(base_url, extra_params=None, page_size=10,
                        delay=0.3, max_retries=3):
    """
    Fetch all pages from a JSONPlaceholder-style endpoint.
    Each page request uses get_with_retry for resilience.
    Throttles between pages with delay.
    """
    params = dict(extra_params or {})
    params['_limit'] = page_size

    all_records, page = [], 1

    while True:
        params['_page'] = page
        batch = get_with_retry(base_url, params=params, max_retries=max_retries)

        if not batch:
            print(f'Empty page {page} — done.')
            break

        all_records.extend(batch)
        print(f'  Page {page:3d}: +{len(batch):3d} records  (total: {len(all_records):4d})')
        page += 1
        time.sleep(delay)

    return all_records

# Fetch all 200 todos from JSONPlaceholder
print('Fetching all todos ...')
all_todos = fetch_all_paginated(
    'https://jsonplaceholder.typicode.com/todos',
    page_size=25,
    delay=0.3,
)
print(f'\nTotal todos fetched: {len(all_todos)}')

# Quick analysis
completed = sum(1 for t in all_todos if t['completed'])
print(f'Completed: {completed}  Pending: {len(all_todos) - completed}')

Fetching all todos ...
  Page   1: + 25 records  (total:   25)
  Page   2: + 25 records  (total:   50)
  Page   3: + 25 records  (total:   75)
  Page   4: + 25 records  (total:  100)
  Page   5: + 25 records  (total:  125)
  Page   6: + 25 records  (total:  150)
  Page   7: + 25 records  (total:  175)
  Page   8: + 25 records  (total:  200)
Empty page 9 — done.

Total todos fetched: 200
Completed: 90  Pending: 110


---
## Quick Reference

| Task | Code |
|------|------|
| GET request | `requests.get(url, params={}, timeout=10)` |
| Parse JSON | `resp.json()` |
| Check status | `resp.raise_for_status()` |
| Status code | `resp.status_code` |
| Retry-After header | `resp.headers.get('Retry-After', 5)` |
| Throttle | `time.sleep(0.5)` |
| Paginate loop | `while True: ... if not batch: break; page += 1` |
| Exponential backoff | `wait = base_delay * (2 ** attempt)` |
| Jitter | `wait += random.uniform(0, 0.5)` |
| Safe nested access | `resp.get('order', {}).get('amount', 0)` |